In [11]:
# Step 1: Install libraries
# Installs necessary Python libraries using pip. These libraries are used for Google API interaction, BigQuery, data manipulation (pandas),
# and working with Excel/CSV files (openpyxl, pyarrow).
!pip install google-api-python-client google-auth google-cloud-bigquery pandas openpyxl pyarrow

In [12]:
# step 2: Import Libraries
# Import specific classes for Google authentication and API interaction.
from google.oauth2 import service_account
from googleapiclient.discovery import build
from google.cloud import bigquery

# Import pandas for data manipulation, requests for making HTTP requests (e.g., downloading files),
# and StringIO/BytesIO for handling in-memory data streams.
import pandas as pd
import requests
from io import StringIO, BytesIO

In [13]:
# Define the path to your service account key file, which contains credentials to authenticate with Google Cloud services.
SERVICE_ACCOUNT_FILE = "/content/financial-dashboard-500409-fb9a3f7a446a.json"
# Define the OAuth 2.0 scopes, which specify the permissions your application needs.
# drive.readonly allows reading from Google Drive, bigquery allows full access to BigQuery.
SCOPES = [
    "https://www.googleapis.com/auth/drive.readonly",
    "https://www.googleapis.com/auth/bigquery"
]

In [14]:
# Load credentials from the service account file and apply the specified scopes.
# These credentials will be used for authenticating with various Google APIs.
credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=SCOPES
)

# Build a Google Drive API service client (version v3) using the loaded credentials.
# This client will be used to interact with Google Drive, e.g., list files in a folder.
drive_service = build("drive", "v3", credentials=credentials)

# Create a BigQuery client using the loaded credentials and specify the Google Cloud project ID.
# This client will be used to interact with Google BigQuery, e.g., load data into tables.
bq_client = bigquery.Client(
    credentials=credentials,
    project="financial-dashboard-500409"
)

# New Section

In [15]:
# Define the Google Drive folder ID from which to fetch files.
# This ID is a unique identifier for a specific folder in Google Drive.
FOLDER_ID = "1kqMyDVr1kD9Upd6490x0i69gLANFomzf"

In [16]:
# Define a function to list files within a given Google Drive folder.
def list_files(service, folder_id):
    # Use the Drive service to list files. The 'q' parameter filters files by parent folder and ensures they are not trashed.
    # 'fields' specifies the file attributes to retrieve (id, name, mimeType).
    # 'pageSize' limits the number of results per request.
    results = service.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields="files(id, name, mimeType)",
        pageSize=1000
    ).execute()
    # Return the list of files found, or an empty list if none are found.
    return results.get("files", [])

# Call the list_files function to get all files from the specified Google Drive folder.
files = list_files(drive_service, FOLDER_ID)

# Iterate through the retrieved files and print their names and MIME types.
# This helps in understanding the content of the folder.
for f in files:
    print(f["name"], f["mimeType"])

combined_part_5.xlsx application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
combined_part_6.xlsx application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
combined_part_4.xlsx application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
combined_part_3.xlsx application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
combined_part_2.xlsx application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
combined_part_1.xlsx application/vnd.openxmlformats-officedocument.spreadsheetml.sheet


In [17]:
file_dataframes = []   # reset every time this cell runs — this line MUST be here

for file in files:
    file_id = file["id"]
    file_name = file["name"]
    mime_type = file["mimeType"]
    print("Reading:", file_name)

    if mime_type == "application/vnd.google-apps.spreadsheet":
        download_url = f"https://docs.google.com/spreadsheets/d/{file_id}/export?format=csv"
        response = requests.get(download_url)
        df = pd.read_csv(StringIO(response.content.decode("utf-8")))
    elif mime_type == "text/csv":
        download_url = f"https://drive.google.com/uc?export=download&id={file_id}"
        response = requests.get(download_url)
        df = pd.read_csv(StringIO(response.content.decode("utf-8")))
    elif mime_type == "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet":
        download_url = f"https://drive.google.com/uc?export=download&id={file_id}"
        response = requests.get(download_url)
        df = pd.read_excel(BytesIO(response.content))
    else:
        print("Skipped:", file_name)
        continue

    df["source_file"] = file_name
    file_dataframes.append(df)

combined_df = pd.concat(file_dataframes, ignore_index=True)

# SAFETY CHECK — add this right after concat
print(f"Total rows after combining: {len(combined_df)}")

# assert len(combined_df) == 150000, f"Row count mismatch! Got {len(combined_df)} rows, expected 150000. Did you re-run a cell?"

Reading: combined_part_5.xlsx
Reading: combined_part_6.xlsx
Reading: combined_part_4.xlsx
Reading: combined_part_3.xlsx
Reading: combined_part_2.xlsx
Reading: combined_part_1.xlsx
Total rows after combining: 36000


In [18]:
# Display the entire combined_df DataFrame. This will show all rows and columns.
combined_df

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,source_file
0,0x12f4a,CUS_0xa08e,September,Philipw,42,#F%$D@*&8,_______,18202.235,1259.852917,2,...,1346.83,37.524379,22 Years and 9 Months,No,0.000000,34.15592067184062,Low_spent_Large_value_payments,361.829371,NaN,combined_part_5.xlsx
1,0x12f4b,CUS_0xa08e,October,Philipw,43,168-64-8956,Doctor,18202.235,1259.852917,2,...,1346.83,27.941043,22 Years and 10 Months,No,0.000000,49.77625366723269,Low_spent_Medium_value_payments,356.209038,NaN,combined_part_5.xlsx
2,0x12f4c,CUS_0xa08e,November,Philipw,43,168-64-8956,Doctor,18202.235,1259.852917,2,...,1346.83,32.943555,22 Years and 11 Months,No,0.000000,109.3222152348699,!@9#%8,286.663076,NaN,combined_part_5.xlsx
3,0x12f4d,CUS_0xa08e,December,Philipw,43,168-64-8956,Doctor,18202.235,NaN,2,...,1346.83,36.719906,23 Years and 0 Months,No,0.000000,132.62483256644293,Low_spent_Small_value_payments,283.360459,NaN,combined_part_5.xlsx
4,0x12f56,CUS_0x4a0d,September,Luciaw,27,129-01-2521,Scientist,35038.35_,3156.862500,4,...,362.03,33.033232,33 Years and 7 Months,Yes,64.245298,__10000__,High_spent_Small_value_payments,367.524043,NaN,combined_part_5.xlsx
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35995,0x5c45,CUS_0x8a8c,December,Karen Jacobsu,32,520-12-7956,Doctor,14219.02,NaN,10,...,3628.5,27.975839,3 Years and 8 Months,NM,76.486008,79.3613953137231,Low_spent_Medium_value_payments,236.84443,NaN,combined_part_1.xlsx
35996,0x5c4e,CUS_0x122f,September,Donovang,24,179-85-5358,Media_Manager,18231.52,NaN,6,...,1358.96,26.281263,15 Years and 6 Months,Yes,86.608647,154.47017169345767,Low_spent_Small_value_payments,193.750515,NaN,combined_part_1.xlsx
35997,0x5c4f,CUS_0x122f,October,Donovang,24,179-85-5358,Media_Manager,18231.52,1448.293333,6,...,1358.96,33.889744,15 Years and 7 Months,Yes,86.608647,42.358067683971626,High_spent_Medium_value_payments,265.862619,NaN,combined_part_1.xlsx
35998,0x5c50,CUS_0x122f,November,NaN,24,179-85-5358,Media_Manager,18231.52,1448.293333,6,...,1358.96,34.718260,15 Years and 8 Months,Yes,86.608647,172.55816847802882,Low_spent_Small_value_payments,175.662518,NaN,combined_part_1.xlsx


In [22]:
from google.cloud import bigquery

PROJECT_ID = "financial-dashboard-500409"
DATASET_ID = "financial_dashboard"
TABLE_ID = "raw_financial_data"

table_ref = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

# Convert 'Monthly_Balance' to numeric, coercing errors to NaN
combined_df["Monthly_Balance"] = pd.to_numeric(combined_df["Monthly_Balance"], errors="coerce")

# Create BigQuery dataset if it doesn't exist
try:
    bq_client.get_dataset(DATASET_ID)
    print(f"Dataset {DATASET_ID} already exists.")
except Exception:
    dataset = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
    dataset.location = "US" # You can specify a different location if needed
    dataset = bq_client.create_dataset(dataset, timeout=30)
    print(f"Dataset {DATASET_ID} created.")

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
    autodetect=True
)

job = bq_client.load_table_from_dataframe(
    combined_df,
    table_ref,
    job_config=job_config
)

job.result()

print("Uploaded to BigQuery:", table_ref)

Dataset financial_dashboard already exists.
Uploaded to BigQuery: financial-dashboard-500409.financial_dashboard.raw_financial_data


In [21]:
print(combined_df.shape)

(36000, 29)
